# Which vectorization?

In [6]:
!pip install mlflow boto3 awscli

In [7]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/")

In [8]:
# Set or create an experiment
mlflow.set_experiment("Exp 2 - BoW vs TfIdf")

<Experiment: artifact_location='s3://ytmlflow-bucket-2026/1', creation_time=1773126824143, experiment_id='1', last_update_time=1773126824143, lifecycle_stage='active', name='Exp 2 - BoW vs TfIdf', tags={}, workspace='default'>

In [9]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [10]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [14]:
import pickle
import os
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns

def run_experiment(vectorizer_type, ngram_range, vectorizer_max_features, vectorizer_name):
    if vectorizer_type == "BoW":
        vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)
    else:
        vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=vectorizer_max_features)

    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_comment'], df['category'], 
        test_size=0.2, random_state=42, stratify=df['category']
    )

    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    with mlflow.start_run():
        mlflow.set_tag("mlflow.runName", f"{vectorizer_name}_{ngram_range}_RandomForest")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")
        mlflow.set_tag("description", f"RandomForest with {vectorizer_name}, ngram_range={ngram_range}, max_features={vectorizer_max_features}")

        mlflow.log_param("vectorizer_type", vectorizer_type)
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", vectorizer_max_features)

        n_estimators = 200
        max_depth = 15
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: {vectorizer_name}, {ngram_range}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Save model as pickle and log as artifact
        model_filename = f"model_{vectorizer_name}_{ngram_range}.pkl"
        with open(model_filename, "wb") as f:
            pickle.dump(model, f)
        mlflow.log_artifact(model_filename)
        os.remove(model_filename)

        print(f"✅ Done: {vectorizer_name}_{ngram_range} | Accuracy: {accuracy:.4f}")


# Run experiments
ngram_ranges = [(1, 1), (1, 2), (1, 3)]
max_features = 5000

for ngram_range in ngram_ranges:
    run_experiment("BoW", ngram_range, max_features, vectorizer_name="BoW")
    run_experiment("TF-IDF", ngram_range, max_features, vectorizer_name="TF-IDF")

✅ Done: BoW_(1, 1) | Accuracy: 0.6480
🏃 View run BoW_(1, 1)_RandomForest at: http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/#/experiments/1/runs/17b9889b51994a4a8425dd0cb8d3fa59
🧪 View experiment at: http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/#/experiments/1
✅ Done: TF-IDF_(1, 1) | Accuracy: 0.6491
🏃 View run TF-IDF_(1, 1)_RandomForest at: http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/#/experiments/1/runs/ca86a61509e04eee9152c6483e532072
🧪 View experiment at: http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/#/experiments/1
✅ Done: BoW_(1, 2) | Accuracy: 0.6569
🏃 View run BoW_(1, 2)_RandomForest at: http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/#/experiments/1/runs/47b3698639b14a9687b12a6693c9f584
🧪 View experiment at: http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/#/experiments/1
✅ Done: TF-IDF_(1, 2) | Accuracy: 0.6568
🏃 View run TF-IDF_(1, 2)_RandomForest at: http://ec2-18-233-223-218.compute-1.amazonaws.com:5000/#/experiments/1/run